In [152]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
import numpy as np
import osmnx as ox
import matplotlib.pyplot as plt

In [ ]:
def join_signal_features(buffered_gdf, feature_gdf, column_name):
    acc_with_feature = gpd.sjoin(buffered_gdf.drop_duplicates('Num_Acc_x'), feature_gdf.to_crs('EPSG:3035'))
    buffered_gdf[column_name] = 'no'
    buffered_gdf.loc[buffered_gdf['Num_Acc_x'].isin(acc_with_feature['Num_Acc_x']), column_name] = 'yes'
    return buffered_gdf


# Table des matières

- [Caractéristiques](#characteristics)
- [Véhicles](#v-hicles)
- [Lieux](#lieux)
- [Usagers](#Usagers)
- [Performing spatial jointure for Lyon](#performing-spatial-jointure-for-lyon)
- [Performing spatial jointure for Paris](#performing-spatial-jointure-for-paris)


# Characteristics

In [ ]:
carac2023 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/104dbb32-704f-4e99-a71e-43563cb604f2',sep=';')
carac2022 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/5fc299c0-4598-4c29-b74c-6a67b0cc27e7',sep=';')
carac2020 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/07a88205-83c1-4123-a993-cba5331e8ae0',sep=';')
carac2021 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/85cfdc0c-23e4-4674-9bcd-79a970d7269b',sep=';')
carac2019 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/e22ba475-45a3-46ac-a0f7-9ca9ed1e283a',sep=';')


In [154]:
carac2022.rename(columns={'Accident_Id': 'Num_Acc'}, inplace=True)

In [155]:
carac=pd.concat([carac2022,carac2020,carac2021,carac2019,carac2023],axis=0)

In [156]:
# Convert 'Accident_Id' values to integers
carac['Num_Acc'] = carac['Num_Acc'].astype(str).replace(" ", "").str.replace(',', '.').astype(float)


In [157]:
carac['lat']=carac['lat'].str.replace(',', '.').astype(float)
carac['long']=carac['long'].str.replace(',', '.').astype(float)

In [158]:
paris=gpd.read_file('https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/georef-france-epci/exports/geojson?lang=fr&refine=epci_name%3A%22M%C3%A9tropole%20du%20Grand%20Paris%22&facet=facet(name%3D%22epci_name%22%2C%20disjunctive%3Dtrue)&timezone=Europe%2FBerlin')

Skipping field reg_code: unsupported OGR type: 5
Skipping field reg_name: unsupported OGR type: 5
Skipping field dep_code: unsupported OGR type: 5
Skipping field dep_name: unsupported OGR type: 5
Skipping field epci_code: unsupported OGR type: 5
Skipping field epci_current_code: unsupported OGR type: 5
Skipping field epci_name: unsupported OGR type: 5


In [159]:
lyon=gpd.read_file('https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/georef-france-epci/exports/geojson?lang=fr&refine=epci_name%3A%22M%C3%A9tropole%20de%20Lyon%22&facet=facet(name%3D%22epci_name%22%2C%20disjunctive%3Dtrue)&timezone=Europe%2FBerlin')

Skipping field reg_code: unsupported OGR type: 5
Skipping field reg_name: unsupported OGR type: 5
Skipping field dep_code: unsupported OGR type: 5
Skipping field dep_name: unsupported OGR type: 5
Skipping field epci_code: unsupported OGR type: 5
Skipping field epci_current_code: unsupported OGR type: 5
Skipping field epci_name: unsupported OGR type: 5


In [160]:
geometry = [Point(xy) for xy in zip(carac['long'], carac['lat'])]

# Créez le GeoDataFrame en spécifiant la colonne de géométrie
carac = gpd.GeoDataFrame(carac, geometry=geometry, crs='EPSG:4326')




# Vehicles

In [ ]:
vehi2022 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/c9742921-4427-41e5-81bc-f13af8bc31a0',sep=';')
vehi2021 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/0bb5953a-25d8-46f8-8c25-b5c2f5ba905e',sep=';')
vehi2020 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/a66be22f-c346-49af-b196-71df24702250',sep=';')
vehi2019 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/780cd335-5048-4bd6-a841-105b44eb2667',sep=';')
vehi2023 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/146a42f5-19f0-4b3e-a887-5cd8fbef057b',sep=';')


In [162]:
vehi=pd.concat([vehi2022,vehi2020,vehi2021,vehi2019,vehi2023],axis=0)

In [163]:
vehi['Num_Acc']=vehi['Num_Acc'].astype(float)

# Zones

In [ ]:
lieux2022 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/a6ef711a-1f03-44cb-921a-0ce8ec975995',sep=';')
lieux2020 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/e85c41f7-d4ea-4faf-877f-ab69a620ce21',sep=';')
lieux2021 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/8a4935aa-38cd-43af-bf10-0209d6d17434',sep=';')
lieux2019 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/2ad65965-36a1-4452-9c08-61a6c874e3e6',sep=';')
lieux2023 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/8bef19bf-a5e4-46b3-b5f9-a145da4686bc',sep=';')


/var/folders/y0/0nrj3m412p978185q3d2503sr02q24/T/ipykernel_97237/2609359383.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  lieux2022 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/a6ef711a-1f03-44cb-921a-0ce8ec975995',sep=';')
/var/folders/y0/0nrj3m412p978185q3d2503sr02q24/T/ipykernel_97237/2609359383.py:5: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  lieux2023 = pd.read_csv( 'https://static.data.gouv.fr/resources/bases-de-donnees-annuelles-des-accidents-corporels-de-la-circulation-routiere-annees-de-2005-a-2023/20241023-153219/lieux-2023.csv',sep=';')


In [165]:
lieux2023=lieux2023.drop_duplicates('Num_Acc')

In [166]:
lieux=pd.concat([lieux2022,lieux2020,lieux2021,lieux2019,lieux2023],axis=0)

# Users

In [ ]:
usagers2022 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/62c20524-d442-46f5-bfd8-982c59763ec8',sep=';')
usagers2020 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/78c45763-d170-4d51-a881-e3147802d7ee',sep=';')
usagers2021 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/ba5a1956-7e82-41b7-a602-89d7dd484d7a',sep=';')
usagers2019 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/36b1b7b3-84b4-4901-9163-59ae8a9e3028',sep=';')
usagers2023 = pd.read_csv( 'https://www.data.gouv.fr/api/1/datasets/r/68848e2a-28dd-4efc-9d5f-d512f7dbe66f',sep=';')

In [168]:
usagers=pd.concat([usagers2022,usagers2020,usagers2021,usagers2019,usagers2023],axis=0)

In [169]:
carac3 = pd.merge(carac,lieux, on='Num_Acc')


In [170]:
carac3 = pd.merge(carac3, vehi, on='Num_Acc')


In [171]:
carac3 = pd.merge(carac3,usagers, on='id_vehicule')


In [172]:
carac3

,Num_Acc_x,jour,mois,an,hrmn,lum,dep,com,agg,int,...,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
0,2.022000e+11,19,10,2022,16:15,1,26,26198,2,3,...,3,1,2008.0,5,2,8,-1,-1,-1,-1
1,2.022000e+11,19,10,2022,16:15,1,26,26198,2,3,...,1,1,1948.0,5,1,8,-1,-1,-1,-1
2,2.022000e+11,20,10,2022,08:34,1,25,25204,2,3,...,4,1,1988.0,9,1,0,-1,0,0,-1
3,2.022000e+11,20,10,2022,08:34,1,25,25204,2,3,...,1,1,1970.0,4,1,0,-1,0,0,-1
4,2.022000e+11,20,10,2022,17:15,1,22,22360,2,6,...,1,1,2002.0,0,1,0,-1,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
619966,2.023001e+11,26,10,2023,17:20,1,973,97302,1,6,...,4,1,1971.0,1,2,0,-1,0,0,-1
619967,2.023001e+11,26,10,2023,17:20,1,973,97302,1,6,...,1,2,1968.0,1,1,0,-1,0,0,-1
619968,2.023001e+11,20,10,2023,16:30,1,69,69387,2,1,...,1,2,2003.0,2,1,-1,-1,-1,-1,-1
619969,2.023001e+11,20,10,2023,16:30,1,69,69387,2,1,...,1,2,2002.0,1,1,-1,-1,-1,-1,-1


In [173]:
data_lyon=gpd.sjoin(carac3,lyon[['epci_name_upper','geometry']]).rename(columns={'epci_name_upper': 'epci'})
data_paris=gpd.sjoin(carac3,paris[['epci_name_upper','geometry']]).rename(columns={'epci_name_upper': 'epci'})


In [174]:
data_lyon = data_lyon.drop(columns=['num_veh_y','num_veh_x','Num_Acc_y','index_right'])
data_paris = data_paris.drop(columns=['num_veh_y','num_veh_x','Num_Acc_y','index_right'])


In [176]:
data_concat=pd.concat([data_paris,data_lyon])

In [177]:
data=data_concat.loc[(data_concat['catu']==1) | (data_concat['catu']==3)]

In [178]:
data['Vehicle']=''
data.loc[(data['catv'] == 7) | (data['catv'] == 10), 'Vehicle'] = 'Car'
data.loc[data['catv'] == 50, 'Vehicle'] = 'E-scooter'
data.loc[data['catv'] == 80, 'Vehicle'] = 'E-bike'
data.loc[data['catv'] == 1, 'Vehicle'] = 'Bike'
data.loc[(data['catv'].isin([30, 31, 32, 33, 34,2])), 'Vehicle'] = 'Motorcycle'
data.loc[data['catv'].isin([13, 14]), 'Vehicle'] = 'Truck'
data.loc[(data['catv'].isin([37, 38])), 'Vehicle'] = 'Bus'
data.loc[(data['catv'] == 40) , 'Vehicle'] = 'Tram'
data.loc[data['catv'].isin([41, 42, 43]), 'Vehicle'] = 'Three-wheeled motorized'
data.loc[data['catv'] == 60, 'Vehicle'] = 'Mechanical PMD'
data.loc[data['catu'] == 3, 'Vehicle'] = 'Pedestrian'

/opt/anaconda3/envs/tfe/lib/python3.13/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [179]:
data = data[data['Vehicle'] != '']


In [180]:
result=data.groupby('Num_Acc_x')['Vehicle'].apply(lambda x: ', '.join(x)).reset_index()

filtered_result = result[result['Vehicle'].str.contains('e-scooter|ebike|bike', case=False)]


In [181]:
filtered_result.rename(columns={'Vehicle': 'Accident'}, inplace=True)


/var/folders/y0/0nrj3m412p978185q3d2503sr02q24/T/ipykernel_97237/3730539010.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_result.rename(columns={'Vehicle': 'Accident'}, inplace=True)


In [182]:
data2=pd.merge(data,filtered_result,on='Num_Acc_x')

In [183]:
def replace_vehicle(row):
    accident_list = row['Accident'].split(', ')
    # Enlever le véhicule principal
    accident_list.remove(row['Vehicle'])
    # Rejoindre les véhicules restants dans une chaîne
    return ', '.join(accident_list)

# Appliquer la fonction à chaque ligne
data2['Involved vehicles'] = data2.apply(replace_vehicle, axis=1)
data2['Involved vehicles'] = data2['Involved vehicles'].replace('',np.nan)

data2['number of involved vehicles'] = data2['Involved vehicles'].apply(lambda x: len(x.split(', ')) if pd.notna(x) else 0) + 1
data2=data2.drop(data2.loc[data2['number of involved vehicles'] > 5].index)

# Séparer la colonne 'Involved vehicles' en plusieurs colonnes
vehicles_split = data2['Involved vehicles'].str.split(', ', expand=True)

# Ajouter des noms de colonnes appropriés
vehicles_split.columns = [f'Involved vehicle {i+1}' for i in range(vehicles_split.shape[1])]

# Ajouter une colonne 'number of involved vehicles'

# Concaténer les colonnes séparées avec la DataFrame originale
data_acci = pd.concat([data2.drop(columns=['Involved vehicles']), vehicles_split], axis=1)

In [184]:
data_acci_selected = data_acci[data_acci['Vehicle'] != 'Pedestrian'][['id_vehicule', 'Vehicle', 'Accident',
                                                                      'number of involved vehicles',
                                                                      'Involved vehicle 1', 'Involved vehicle 2',
                                                                      'Involved vehicle 3', 'Involved vehicle 4']]


# Fusion des DataFrames sur la colonne 'id_vehicule'
data_acci = pd.merge(data_concat, data_acci_selected, on='id_vehicule')
data_acci.loc[data_acci['catu'] == 3, 'Vehicle'] = 'Pedestrian'

# Performing spatial jointure for Lyon

In [185]:
accidents_lyon=data_acci.loc[data_acci['epci']== 'MÉTROPOLE DE LYON']
accidents_lyon=accidents_lyon[['Num_Acc_x','int','geometry','adr']].drop_duplicates('Num_Acc_x')

In [186]:
chaussees=gpd.read_file('lyon_network.gpkg')
#chaussees=gpd.read_file('https://data.grandlyon.com/geoserver/metropole-de-lyon/ows?SERVICE=WFS&VERSION=2.0.0&request=GetFeature&typename=metropole-de-lyon:pvo_patrimoine_voirie.pvochausseetrottoir&outputFormat=application/json&SRSNAME=EPSG:4171&sortBy=gid')

In [187]:
chaussees.to_crs(epsg=3035, inplace=True)

In [188]:
accidents_lyon=accidents_lyon.to_crs(chaussees.crs)
distances = accidents_lyon.geometry.apply(lambda point: chaussees.distance(point))
dist_transpose=np.transpose(distances)

In [189]:
nearest_indices = dist_transpose.apply(lambda dist: dist.idxmin())

# Assign attributes of nearest features to accident points
nearest_features = chaussees.iloc[nearest_indices]  # Flatten indices to 1D array for indexing
nearest_features_reset  = nearest_features.reset_index(drop=True)

nearest_features = nearest_features.drop(columns='geometry')

accidents_lyon = pd.concat([accidents_lyon.reset_index(drop=True), nearest_features.reset_index(drop=True)], axis=1)


In [190]:
# Définir le polygone de la Métropole de Lyon
lyon_metropole_polygon = ox.geocode_to_gdf("Métropole de Lyon, France")

# Récupérer les données pour toute la métropole
lyon_traffic_signals = ox.features_from_polygon(
    lyon_metropole_polygon.geometry.iloc[0],
    tags={"highway": "traffic_signals"}
)
lyon_stops = ox.features_from_polygon(
    lyon_metropole_polygon.geometry.iloc[0],
    tags={"highway": "stop"}
)
lyon_give_way = ox.features_from_polygon(
    lyon_metropole_polygon.geometry.iloc[0],
    tags={"highway": "give_way"}
)

# Conversion du CRS
lyon_traffic_signals.to_crs(epsg=3035, inplace=True)
lyon_stops.to_crs(epsg=3035, inplace=True)
lyon_give_way.to_crs(epsg=3035, inplace=True)

In [191]:
accidents_lyon=accidents_lyon.to_crs('EPSG:3035')
acc_lyon_int=accidents_lyon.loc[(accidents_lyon['int']==2) | (accidents_lyon['int']== 3) | (accidents_lyon['int']==4) | (accidents_lyon['int']==5) ].drop_duplicates('Num_Acc_x')
acc_lyon_buffered = acc_lyon_int.to_crs('EPSG:3035').buffer(25)

acc_lyon_buffered_gdf = gpd.GeoDataFrame(acc_lyon_int,geometry=acc_lyon_buffered, crs='EPSG:3035')


acc_lyon_buffered_gdf = join_signal_features(acc_lyon_buffered_gdf, lyon_stops, 'stops')
acc_lyon_buffered_gdf = join_signal_features(acc_lyon_buffered_gdf, lyon_give_way, 'give_way')
acc_lyon_buffered_gdf = join_signal_features(acc_lyon_buffered_gdf, lyon_traffic_signals, 'traffic_lights')
accidents_lyon=pd.merge(accidents_lyon,acc_lyon_buffered_gdf[['Num_Acc_x','stops','give_way','traffic_lights']].drop_duplicates('Num_Acc_x'),on='Num_Acc_x',how='left')


# Performing spatial jointure for Paris

In [192]:
accidents_paris=data_acci.loc[data_acci['epci']== 'MÉTROPOLE DU GRAND PARIS']
accidents_paris=accidents_paris[['Num_Acc_x','int','geometry']].drop_duplicates('Num_Acc_x')


In [193]:
reseau_cyclable=gpd.read_file('https://data.iledefrance.fr/api/explore/v2.1/catalog/datasets/amenagements-velo-en-ile-de-france/exports/geojson?lang=fr&timezone=Europe%2FBerlin').to_crs('EPSG:2154')

In [194]:
paris.to_crs('EPSG:2154',inplace=True)
reseau_cyclable_paris=gpd.sjoin(reseau_cyclable,paris[['geometry','epci_name_upper']])

In [195]:
accidents_paris.to_crs('EPSG:3035',inplace=True)
reseau_cyclable_paris.to_crs('EPSG:3035',inplace=True)

In [196]:
distances_paris = accidents_paris.geometry.apply(lambda point: reseau_cyclable_paris.distance(point))


/opt/anaconda3/envs/tfe/lib/python3.13/site-packages/shapely/measurement.py:72: RuntimeWarning: invalid value encountered in distance
  return lib.distance(a, b, **kwargs)


In [197]:
# Transpose the distances DataFrame
dist_transpose_paris = np.transpose(distances_paris)

# Find the index of the nearest cycling path for each accident point
nearest_indices = dist_transpose_paris.apply(lambda dist: dist.idxmin())


In [198]:
# Extract the attributes of the nearest cycling paths
nearest_features = reseau_cyclable.iloc[nearest_indices]


In [199]:
# Reset the index of nearest_features to align with the indices of accident_sur_piste
nearest_features_reset = nearest_features.reset_index(drop=True)

nearest_features = nearest_features.drop(columns='geometry')


In [200]:
# Assign the nearest cycling path ID to each accident point
accidents_paris = pd.concat([accidents_paris.reset_index(drop=True), nearest_features.reset_index(drop=True)], axis=1)

In [201]:
reseau_cyclable_paris.to_crs('EPSG:4326').to_file('reseau_cyclable_paris.geojson')

In [202]:
# Définir le polygone de la Métropole du Grand Paris
grand_paris_polygon = ox.geocode_to_gdf("Métropole du Grand Paris, France")

# Récupérer les données pour toute la métropole
paris_traffic_signals = ox.features_from_polygon(
    grand_paris_polygon.geometry.iloc[0],
    tags={"highway": "traffic_signals"}
)
paris_stops = ox.features_from_polygon(
    grand_paris_polygon.geometry.iloc[0],
    tags={"highway": "stop"}
)
paris_give_way = ox.features_from_polygon(
    grand_paris_polygon.geometry.iloc[0],
    tags={"highway": "give_way"}
)

# Conversion du CRS
paris_traffic_signals.to_crs(epsg=3035, inplace=True)
paris_stops.to_crs(epsg=3035, inplace=True)
paris_give_way.to_crs(epsg=3035, inplace=True)

In [203]:
acc_paris_int=accidents_paris.loc[(accidents_paris['int']==2) | (accidents_paris['int']== 3) | (accidents_paris['int']==4) | (accidents_paris['int']==5)].drop_duplicates('Num_Acc_x')

In [204]:
acc_buffered = acc_paris_int.to_crs('EPSG:3035').buffer(25)

acc_buffered_gdf = gpd.GeoDataFrame(acc_paris_int,geometry=acc_buffered, crs='EPSG:3035')

In [205]:
acc_buffered_gdf = join_signal_features(acc_buffered_gdf, paris_stops, 'stops')


In [206]:
acc_buffered_gdf = join_signal_features(acc_buffered_gdf, paris_give_way, 'give_way')


In [207]:
acc_buffered_gdf = join_signal_features(acc_buffered_gdf, paris_traffic_signals, 'traffic_lights')


In [208]:
accidents_paris=pd.merge(accidents_paris,acc_buffered_gdf[['Num_Acc_x','stops','give_way','traffic_lights']].drop_duplicates('Num_Acc_x'),on='Num_Acc_x',how='left')


In [209]:
accidents_lyon=accidents_lyon.drop(columns='adr')

In [210]:
accidents_lyon.to_crs('EPSG:4326').to_file('acci_lyon.geojson')

In [105]:
accidents_paris.to_crs('EPSG:4326').to_file('acci_paris.geojson')


In [11]:
accidents_lyon=gpd.read_file('acci_lyon.geojson')
accidents_paris=gpd.read_file('acci_paris.geojson')
data_lyon=gpd.read_file('data_lyon.gpkg')
data_paris=gpd.read_file('data_paris.gpkg')

In [12]:
paris=pd.merge(data_paris,accidents_paris.drop(columns=['int','geometry']),on='Num_Acc_x',how='left')
lyon=pd.merge(data_lyon,accidents_lyon.drop(columns=['int','geometry']),on='Num_Acc_x',how='left')
paris.rename(columns={'Num_Acc_x':'Num_Acc'},inplace=True)
lyon.rename(columns={'Num_Acc_x':'Num_Acc'},inplace=True)
paris=paris.rename(columns={'EPCI_name_u':'ville'})

In [13]:
# Combine the data from Paris and Lyon into a single DataFrame
acci = pd.concat([paris, lyon])

In [19]:
acci_clean = acci.drop(columns=[
    'dep','agg','occutc','secu3','v2','v1','pr1','pr','lartpc','larrout',
    'geo_shape','panneaux','revetementpiste','limitatio2','limitatio1',
    'limitatio3','limitatio0','osm_id','petite_ech',
    'commune2_left','insee2_left','nomvoie2','denominati','domanialit', 'notes','precisionr','continuite','matieresda','itineraire',
    'sens_voit','codefuv','insee1_left','insee1_right','gid_right','itinerair0',
    'commune1_left','commune1_right','nomvoie1','voie','insee_com','codetronco','geo_point_2d','hierarchi0','revetemen0','revetemen1','limitation',
    'hierarchie','routegrand','id','nom','observation','zonecirculationapaisee','commune2_right','insee2_right','financementac','typeamenagement2',
])

In [20]:
def clean_data(acci_clean):
    # Replace all instances of "Sens Unique" with "UNIQUE" in column: 'senscircul'
    acci_clean['senscirculation'] = acci_clean['senscirculation'].str.replace("Sens Unique", "UNIQUE", case=False, regex=False)
    # Replace all instances of "double" with "DOUBLE" in column: 'senscircul'
    acci_clean['senscirculation'] = acci_clean['senscirculation'].str.replace("double", "DOUBLE", case=False, regex=False)
    # Drop column: 'sens'
    return acci_clean

acci_clean = clean_data(acci_clean.copy())

In [ ]:
def rename_columns(acci_clean):
 # Drop column: 'tram_crossing'
    #acci_clean = acci_clean.drop(columns=['tram_crossing'])
    # Replace all instances of "z30" with "Zone 30" in column: 'nv'
    acci_clean['nv'] = acci_clean['nv'].str.replace("z30", "Zone 30", case=False, regex=False)
    # Replace all instances of "z20" with "Zone de rencontre" in column: 'nv'
    acci_clean['nv'] = acci_clean['nv'].str.replace("z20", "Zone de rencontre", case=False, regex=False)
    # Replace all instances of "rue pietonne" with "Aire Piétonne" in column: 'nv'
    acci_clean['nv'] = acci_clean['nv'].str.replace("rue pietonne", "Aire Piétonne", case=False, regex=False)
    
    acci_clean['reglementa'] = acci_clean['reglementa'].combine_first(acci_clean['nv'])
    # Rename column 'longueurtr' to 'longueur_trottoir_droit'
    acci_clean = acci_clean.rename(columns={'longueurtr': 'longueur_trottoir_droit'})
    # Rename column 'surfacetro' to 'surface_trottoir_droit'
    acci_clean = acci_clean.rename(columns={'surfacetro': 'surface_trottoir_droit'})
    # Rename column 'largeurtr0' to 'largeur_trottoir_droit'

    acci_clean = acci_clean.rename(columns={'largeurtr0': 'largeur_trottoir_droit'})

    acci_clean=acci_clean.rename(columns={'reseaupl': 'Truck traffic', 'revetement': 'Pavement','epci': 'Agglomeration'})
    
    # Rename column 'longueurt0' to 'longueur_trottoir_gauche'
    acci_clean = acci_clean.rename(columns={'longueurt0': 'longueur_trottoir_gauche'})
    # Rename column 'surfacetr0' to 'surface_trottoir_gauche'
    acci_clean = acci_clean.rename(columns={'surfacetr0': 'surface_trottoir_gauche'})
    return acci_clean

acci_clean = rename_columns(acci_clean.copy())


,Num_Acc,jour,mois,an,hrmn,lum,com,int,atm,col,...,reseau,typeamenagement,positionnement,senscirculation,environnement,localisation,typologiepiste,domanialite,reglementation,anneelivraison
0,2.022000e+11,21,10,2022,16:32,1,75106,1,1,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.022000e+11,21,10,2022,16:32,1,75106,1,1,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2.022000e+11,20,10,2022,13:00,1,75105,2,1,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2.022000e+11,20,10,2022,13:00,1,75105,2,1,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2.022000e+11,21,10,2022,11:25,1,75113,4,1,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
acci_clean.loc[(acci_clean['int']!=1) ,'Crossroad']='No traffic lights'
acci_clean.loc[(acci_clean['traffic_lights']=='yes'),'Crossroad']='Traffic lights'
acci_clean.loc[(acci_clean['stops']=='yes'),'Crossroad']='Stop sign'
acci_clean.loc[(acci_clean['int']==6),'Crossroad']='Roundabouts'
acci_clean['Crossroad'] = acci_clean['Crossroad'].fillna('No intersection')


In [23]:
acci_clean['Cycle facilities']=''

acci_clean.loc[acci_clean['Agglomeration'].isin(['MÉTROPOLE DU GRAND PARIS','MÉTROPOLE DE LYON']),'Cycle facilities']='No cycle facilities'
acci_clean.loc[acci_clean['moyenn_ech'].isin(['21','22']),'Cycle facilities']='Cycle lane'
acci_clean.loc[acci_clean['moyenn_ech'].isin(['11','12']),'Cycle facilities']='Cycle path/Greenway'
acci_clean.loc[acci_clean['moyenn_ech'].isin(['21','22']),'Cycle facilities']='Cycle lane'
acci_clean.loc[acci_clean['moyenn_ech'].isin(['23','24']),'Cycle facilities']='Bus lane'
acci_clean.loc[acci_clean['moyenn_ech'].isin(['41']),'Cycle facilities']='Pedestrianized street'

acci_clean.loc[acci_clean['typeamenagement'].str.contains('Bande Cyclable', na=False),'Cycle facilities']='Cycle lane'
acci_clean.loc[acci_clean['typeamenagement'].str.contains('Piste Cyclable', na=False),'Cycle facilities']='Cycle path/Greenway'
acci_clean.loc[acci_clean['typeamenagement'].str.contains('Voie verte', na=False),'Cycle facilities']='Cycle path/Greenway'

acci_clean.loc[acci_clean['typeamenagement'].str.contains('bus', na=False),'Cycle facilities']='Bus lane'

In [24]:
acci_clean['positionnement_piste']=acci_clean['positionnement']
acci_clean.loc[(acci_clean['positionnement'].str.contains('unidirectionnel')) &(acci_clean['positionnement'].str.contains('unidirectionnel')),'positionnement_piste']='Unidirectionnel'
acci_clean.loc[(acci_clean['positionnement'].str.contains('bidirectionnel')) &(acci_clean['positionnement'].str.contains('bidirectionnel')),'positionnement_piste']='Bidirectionnel'
acci_clean.loc[(acci_clean['ad'].str.contains('uni')) &(acci_clean['ag'].str.contains('uni')),'positionnement_piste']='Unidirectionnel'
acci_clean.loc[(acci_clean['ad'].str.contains('uni')) | (acci_clean['ag'].str.contains('uni')),'positionnement_piste']='Unidirectionnel'
acci_clean.loc[(acci_clean['ad'].str.contains('piste uni')) &(acci_clean['ag'].str.contains('piste uni')),'positionnement_piste']='Bidirectionnel'
acci_clean.loc[(acci_clean['ad'].str.contains('bi')) | (acci_clean['ag'].str.contains('bi')),'positionnement_piste']='Bidirectionnel'
acci_clean.loc[(acci_clean['ad'].str.contains('DSC')) | (acci_clean['ag'].str.contains('DSC')),'positionnement_piste']='Bidirectionnel'

In [25]:
acci_clean = acci_clean.reset_index(drop=True)

In [26]:
acci_clean.index.name='index'

In [27]:
acci_clean['age']=acci_clean['an']-acci_clean['an_nais']

In [28]:
bins = [0, 20, 40, 60,100]  # Intervalles d'âge
labels = ['0-20', '21-40', '41-60','61+']  # Étiquettes pour chaque intervalle

# Créer une nouvelle colonne 'Age category' contenant les catégories d'âge
acci_clean['Age category'] = pd.cut(acci_clean['age'], bins=bins, labels=labels, right=False).astype(str)

In [29]:
acci_clean['severity'] =acci_clean['grav'].replace({2: 3, 4: 2, 3:2})

In [30]:
def determine_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    elif month in [9, 10, 11]:
        return 'Autumn'
    else:
        return 'Unknown'  # Au cas où il y aurait une valeur invalide

# Appliquer la fonction pour créer la colonne 'season'
acci_clean['season'] = acci_clean['mois'].apply(determine_season)

In [31]:
acci_clean['jour'] = acci_clean['jour'].astype(int)
acci_clean['mois'] = acci_clean['mois'].astype(int)
acci_clean['an'] = acci_clean['an'].astype(int)

In [32]:
acci_clean.rename(columns={'jour':'day','mois':'month','an':'Year'},inplace=True)

In [33]:
acci_clean['date'] = pd.to_datetime(acci_clean[['day', 'month', 'Year']],format='%Y-%m-%d')
acci_clean['date_str'] = acci_clean['Year'].astype(str) + '-' + acci_clean['month'].astype(str).str.zfill(2) + '-01'

# Convertir en type date
acci_clean['date2'] = pd.to_datetime(acci_clean['date_str'], format='%Y-%m-%d')

# Supprimer la colonne temporaire si nécessaire
acci_clean.drop('date_str', axis=1, inplace=True)

# Extraire le jour de la semaine
acci_clean['day_of_week'] = acci_clean['date'].dt.day_name()

In [34]:
acci_clean['Weekend']='No'
acci_clean.loc[acci_clean['day_of_week'].isin(['Sunday', 'Saturday']), 'Weekend'] = 'Yes'

In [35]:
acci_clean['hour'] = acci_clean['hrmn'].apply(lambda x: int(str(x).zfill(4)[:2]))

In [36]:
def time_of_day(hour):
    if 7 <= hour < 9:
        return 'Morning peak hours'

    elif 17 <= hour < 19:
        return 'Afternoon peak hours'
    else:
        return 'Offpeak'
    

acci_clean['Time of day'] = acci_clean['hour'].apply(time_of_day)

In [37]:
acci_clean['Helmet']='No'
acci_clean.loc[(acci_clean['secu1']==2),'Helmet']='Yes'
acci_clean.loc[(acci_clean['secu2']==2),'Helmet']='Yes'

acci_clean['Reflective jacket']='No'
acci_clean.loc[(acci_clean['secu1']==4),'Reflective jacket']='Yes'
acci_clean.loc[(acci_clean['secu2']==4),'Reflective jacket']='Yes'

In [39]:
acci_clean.loc[acci_clean['Pavement'].isin(['Béton Bitumineux','asphalt']), 'Pavement',] = 'Asphalt'
acci_clean.loc[acci_clean['Pavement'].isin(['sett', 'Pavés', 'paved', 'paving_stones', 'Dalles Pavés','cobblestone']), 'Pavement'] = 'Paved'
acci_clean.loc[acci_clean['Pavement'].isin(['concrete']), 'Pavement'] = 'Concrete'

acci_clean.loc[~acci_clean['Pavement'].isin(['Asphalt','Paved','Concrete']), 'Pavement'] = ''
print(acci_clean[['Pavement']].value_counts())

Pavement
Asphalt     22149
             1946
Paved        1354
Concrete      211
Name: count, dtype: int64


In [40]:
acci_clean['Surface condition']=''
acci_clean.loc[acci_clean['surf'].isin([1]), 'Surface condition',] = 'Normal'
acci_clean.loc[acci_clean['surf'].isin([2, 3, 4,6]), 'Surface condition'] = 'Wet'

print( acci_clean['Surface condition'].value_counts())

Surface condition
Normal    21691
Wet        3896
             73
Name: count, dtype: int64


In [41]:
acci_clean['Pedestrian action'] = ''
acci_clean.loc[acci_clean['actp'].isin(['3']), 'Pedestrian action'] = 'Crossing'
acci_clean.loc[acci_clean['actp'].isin(['2','1']), 'Pedestrian action'] = 'Moving'

print( acci_clean['Pedestrian action'].value_counts())

Pedestrian action
            24234
Crossing     1244
Moving        182
Name: count, dtype: int64


In [42]:
acci_clean['Pedestrian localisation']=''
acci_clean.loc[acci_clean['locp'].isin([1,2]), 'Pedestrian localisation',] = 'On the road'
acci_clean.loc[acci_clean['locp'].isin([4, 3]), 'Pedestrian localisation'] = 'On a pedestrian crossing'
acci_clean.loc[acci_clean['locp'].isin([5]), 'Pedestrian localisation'] = 'On sidewalk'
print( acci_clean['Pedestrian localisation'].value_counts())

Pedestrian localisation
                            24082
On a pedestrian crossing      710
On the road                   631
On sidewalk                   237
Name: count, dtype: int64


In [43]:
acci_clean['Point of impact']=''
acci_clean.loc[acci_clean['choc'].isin([1,2,3]), 'Point of impact',] = 'Front'
acci_clean.loc[acci_clean['choc'].isin([4, 5,6]), 'Point of impact'] = 'Back'
acci_clean.loc[acci_clean['choc'].isin([7]), 'Point of impact'] = 'Right side'
acci_clean.loc[acci_clean['choc'].isin([8]), 'Point of impact'] = 'Left side'

acci_clean.loc[acci_clean['choc'].isin([0]), 'Point of impact'] = 'None'


print( acci_clean['Point of impact'].value_counts())

Point of impact
Front         16050
Left side      2553
None           2459
Back           2439
Right side     2115
                 44
Name: count, dtype: int64


In [44]:
acci_clean['Intersection']='Other'
acci_clean.loc[acci_clean['int'].isin([1]), 'Intersection',] = 'No intersection'
acci_clean.loc[acci_clean['int'].isin([2]), 'Intersection'] = 'Cross intersection'
acci_clean.loc[acci_clean['int'].isin([3]), 'Intersection'] = 'T intersection'
acci_clean.loc[acci_clean['int'].isin([4]), 'Intersection'] = 'Y intersection'
acci_clean.loc[acci_clean['int'].isin([6]), 'Intersection'] = 'Roundabout'
acci_clean.loc[acci_clean['int'].isin([7]), 'Intersection'] = 'Square'


In [45]:
acci_clean['Long profile']=''
acci_clean.loc[acci_clean['prof'].isin([-1]), 'Long profile',] = 'Unspecified'
acci_clean.loc[acci_clean['prof'].isin([2,3,4]), 'Long profile'] = 'Slope'
acci_clean.loc[acci_clean['prof'].isin([1]), 'Long profile'] = 'Flat'



In [46]:
acci_clean['Trip purpose']='Other'
acci_clean.loc[acci_clean['trajet'].isin([5,3]), 'Trip purpose'] = 'Leisure/Shopping'
acci_clean.loc[acci_clean['trajet'].isin([2,1,4]), 'Trip purpose'] = 'Professional use'

print( acci_clean['Trip purpose'].value_counts())

Trip purpose
Other               10060
Professional use     8584
Leisure/Shopping     7016
Name: count, dtype: int64


In [47]:
acci_clean['Weather conditions']=''
acci_clean.loc[acci_clean['atm'].isin([1]), 'Weather conditions'] = 'Normal'
acci_clean.loc[acci_clean['atm'].isin([2,3,4]), 'Weather conditions'] = 'Rain / Snow'
acci_clean.loc[acci_clean['atm'].isin([8,5]), 'Weather conditions'] = 'Cloudy / Fog'
print( acci_clean['Weather conditions'].value_counts())

Weather conditions
Normal          21232
Rain / Snow      2870
Cloudy / Fog     1267
                  291
Name: count, dtype: int64


In [48]:
acci_clean['Lighting conditions']=''
acci_clean.loc[acci_clean['lum'].isin([1]), 'Lighting conditions'] = 'Daylight'
acci_clean.loc[acci_clean['lum'].isin([2]), 'Lighting conditions'] = 'Twilight'
acci_clean.loc[acci_clean['lum'].isin([3,4]), 'Lighting conditions'] = 'Night without street lightings'

acci_clean.loc[acci_clean['lum'].isin([5]), 'Lighting conditions'] = 'Night with street lightings on'

print( acci_clean['Lighting conditions'].value_counts())

Lighting conditions
Daylight                          18609
Night with street lightings on     5317
Twilight                           1425
Night without street lightings      309
Name: count, dtype: int64


In [49]:
acci_clean['Gender']=''
acci_clean.loc[acci_clean['sexe'].isin([1]), 'Gender'] = 'Male'
acci_clean.loc[acci_clean['sexe'].isin([2]), 'Gender'] = 'Female'

In [50]:
acci_clean['User category']=''
acci_clean.loc[acci_clean['catu'].isin([1]), 'User category'] = 'Driver'
acci_clean.loc[acci_clean['catu'].isin([2]), 'User category'] = 'Passenger'
acci_clean.loc[acci_clean['catu'].isin([3]), 'User category'] = 'Pedestrian'

In [51]:
acci_clean['Road type']=''
acci_clean.loc[acci_clean['highway'].isin(['residential']), 'Road type'] = 'Residential'
acci_clean.loc[acci_clean['highway'].isin(['primary']), 'Road type'] = 'Primary'
acci_clean.loc[acci_clean['highway'].isin(['secondary']), 'Road type'] = 'Secondary'
acci_clean.loc[acci_clean['highway'].isin(['tertiary']), 'Road type'] = 'Tertiary'
acci_clean.loc[acci_clean['highway'].isin(['cycleway']), 'Road type'] = 'Cycleway'
acci_clean.loc[acci_clean['highway'].isin(['path']), 'Road type'] = 'Path'
acci_clean.loc[acci_clean['highway'].isin(['footway','pedestrian','living_street']), 'Road type'] = 'Pedestrian area'

In [52]:
acci_clean['Road width'] = pd.cut(acci_clean['largeurcha'], bins=[0,4,6,8,10,12,100], labels=['<4 m','4-6 m','6-8 m','8-10 m','10-12 m', '> 12m'], right=False).astype(str)


In [53]:
acci_clean['Vehicle type']=''
acci_clean.loc[acci_clean['Vehicle'].isin(['Truck','Tram','Bus']), 'Vehicle type'] = 'Large motorized vehicle'
acci_clean.loc[acci_clean['Vehicle'].isin(['Car']), 'Vehicle type'] = 'Cars'
acci_clean.loc[acci_clean['Vehicle'].isin(['Motorcycle','Three-wheeled motorized']), 'Vehicle type'] = 'Light motorized vehicle'
acci_clean.loc[acci_clean['Vehicle'].isin(['Pedestrian']), 'Vehicle type'] = 'Pedestrian'
acci_clean.loc[acci_clean['Vehicle'].isin(['Mechanical PMD','Bike', 'E-bike','E-scooter',]), 'Vehicle type'] = 'Micromobility vehicle'

In [54]:
acci_clean['Reglementation']=''
acci_clean.loc[acci_clean['reglementa'].isna(), 'Reglementation'] = 'No particularity'
acci_clean.loc[acci_clean['reglementa'].isin(['Zone 30']), 'Reglementation'] = 'Zone 30'
acci_clean.loc[acci_clean['reglementa'].isin(['Aire Piétonne']), 'Reglementation'] = 'Pedestrian area'
acci_clean.loc[acci_clean['reglementa'].isin(['Zone de rencontre']), 'Reglementation'] = 'Meeting zone (Z20)'
acci_clean.loc[acci_clean['reglementa'].isin(['limite 30']), 'Reglementation'] = 'Limit 30'




In [55]:
acci_clean['Max speed']='Other'
acci_clean.loc[acci_clean['vma'].isin([50]), 'Max speed'] = '50 km/h'
acci_clean.loc[acci_clean['vma'].isin([30]), 'Max speed'] = '30 km/h'
acci_clean.loc[acci_clean['vma'].isin([20]), 'Max speed'] = '20 km/h'
acci_clean.loc[acci_clean['vma'].isin([1,2,3,8,6,5,10]), 'Max speed'] = '< 10 km/h'



In [56]:
# Step 1: Filter for passengers
passenger_data = acci_clean[acci_clean['catu'] == 2]

# Step 2: Count the number of passengers per accident
passenger_counts = passenger_data.groupby('id_vehicule').size().reset_index(name='Number of passengers')

passenger_counts['Number of passengers'] =passenger_counts['Number of passengers'].astype(str)
# Step 3: Merge the counts with the original dataframe
acci_clean = pd.merge(acci_clean, passenger_counts, on='id_vehicule', how='left')

# Fill NaN values with 0 for accidents without passengers
acci_clean['Number of passengers'] = acci_clean['Number of passengers'].fillna(0)

In [57]:
acci_clean['Maneuver']='Other'
acci_clean.loc[acci_clean['manv'].isin([5]), 'Maneuver'] = 'In the opposite direction'
acci_clean.loc[acci_clean['manv'].isin([1,2]), 'Maneuver'] = 'Without change of direction'
acci_clean.loc[acci_clean['manv'].isin([15]), 'Maneuver'] = 'Turning left'
acci_clean.loc[acci_clean['manv'].isin([16]), 'Maneuver'] = 'Turning right'
acci_clean.loc[acci_clean['manv'].isin([24]), 'Maneuver'] = 'Parked'

acci_clean.loc[acci_clean['manv'].isin([19]), 'Maneuver'] = 'Crossing the road'
acci_clean.loc[acci_clean['manv'].isin([22]), 'Maneuver'] = 'Door opening'

acci_clean.loc[acci_clean['manv'].isin([13,14]), 'Maneuver'] = 'Pulled away'
acci_clean.loc[acci_clean['manv'].isin([17,18]), 'Maneuver'] = 'Overtaking'



In [58]:
acci_clean['Obstacle']='Other'
acci_clean.loc[acci_clean['obs'].isin([1]), 'Obstacle'] = 'Parked_vehicle'
acci_clean.loc[acci_clean['obs'].isin([8]), 'Obstacle'] = 'Poteau'
acci_clean.loc[acci_clean['manv'].isin([12]), 'Obstacle'] = 'Sidewalk bordure'




### Second-party vehicle

In [59]:
# Define a danger ranking for the vehicle types
danger_ranking = {
    'Large motorized vehicle': 1,
    'Cars': 2,
    'Light motorized vehicle': 3,
    'Micromobility vehicle':4

}

# Assign the danger ranking to each row
acci_clean['danger_rank'] = acci_clean['Vehicle type'].map(danger_ranking)
acci_clean.loc[acci_clean['Vehicle'] == 'Pedestrian', 'id_vehicule'] = range(len(acci_clean.loc[acci_clean['Vehicle'] == 'Pedestrian']))
# Perform the initial self-merge

In [60]:

driver=acci_clean.loc[acci_clean['catu']!=2]
df_merged = pd.merge(driver, driver, on='Num_Acc', suffixes=('', '_opposite'))

# Filter out rows where the id_vehicule is the same
df_filtered = df_merged[df_merged['id_vehicule'] != df_merged['id_vehicule_opposite']]

# Select relevant columns
df_filtered = df_filtered[['Num_Acc', 'id_vehicule', 'id_vehicule_opposite','age_opposite', 'Vehicle_opposite', 'Vehicle type_opposite', 'Maneuver_opposite', 'Gender_opposite', 'severity_opposite', 'Point of impact_opposite']]

# Rename columns for clarity
df_filtered.rename(columns={
    'age_opposite': 'age_2', 
    'Vehicle_opposite': 'Vehicle_2',
    'Vehicle type_opposite': 'vehicle_type_2', 
    'Maneuver_opposite': 'Maneuver_2',
    'Gender_opposite': 'Gender_2',
    'severity_opposite': 'severity_2',
    'Point of impact_opposite': 'Point of impact_2',
    'id_vehicule_opposite': 'id_vehicule_opposite_2'
}, inplace=True)

# Assign danger ranking to the opposite vehicle type
df_filtered['danger_rank_2'] = df_filtered['vehicle_type_2'].map(danger_ranking)

# Sort by Num_Acc, id_vehicule, and danger_rank_2
df_filtered = df_filtered.sort_values(by=['Num_Acc', 'id_vehicule', 'danger_rank_2'])

# Drop duplicates to keep the row with the most dangerous vehicle
df_filtered = df_filtered.drop_duplicates(subset=['Num_Acc', 'id_vehicule'])

# Merge back with the original dataframe to include age_2, vehicle_type_2, and Maneuver_2
acci_clean = pd.merge(acci_clean, df_filtered[['Num_Acc', 'id_vehicule', 'age_2', 'vehicle_type_2', 'Vehicle_2', 'Maneuver_2', 'Gender_2', 'severity_2', 'Point of impact_2','id_vehicule_opposite_2']], on=['Num_Acc', 'id_vehicule'], how='left')

In [61]:

# Repeat the process for the third vehicle
driver=acci_clean.loc[acci_clean['catu']!=2]

df_merged = pd.merge(driver, driver, on='Num_Acc', suffixes=('', '_opposite'))

df_filtered_3 = df_merged[(df_merged['id_vehicule'] != df_merged['id_vehicule_opposite'])]


df_filtered_3 = df_filtered_3[df_filtered_3['id_vehicule'] != df_filtered_3['id_vehicule_opposite']]
df_filtered_3 = df_filtered_3[ (df_filtered_3['id_vehicule_opposite_2'] != df_filtered_3['id_vehicule_opposite'])]

df_filtered_3 = df_filtered_3[['Num_Acc', 'id_vehicule', 'id_vehicule_opposite','age_opposite', 'Vehicle_opposite', 'Vehicle type_opposite', 'Maneuver_opposite', 'Gender_opposite', 'severity_opposite', 'Point of impact_opposite']]
df_filtered_3.rename(columns={
    'age_opposite': 'age_3', 
    'Vehicle_opposite': 'Vehicle_3',
    'Vehicle type_opposite': 'vehicle_type_3', 
    'Maneuver_opposite': 'Maneuver_3',
    'Gender_opposite': 'Gender_3',
    'severity_opposite': 'severity_3',
    'Point of impact_opposite': 'Point of impact_3',
    'id_vehicule_opposite': 'id_vehicule_opposite_3'
}, inplace=True)
df_filtered_3['danger_rank_3'] = df_filtered_3['vehicle_type_3'].map(danger_ranking)
df_filtered_3 = df_filtered_3.sort_values(by=['Num_Acc', 'id_vehicule', 'danger_rank_3'])
df_filtered_3 = df_filtered_3.drop_duplicates(subset=['Num_Acc', 'id_vehicule'])
acci_clean = pd.merge(acci_clean, df_filtered_3[['Num_Acc', 'id_vehicule','age_3', 'vehicle_type_3', 'Vehicle_3', 'Maneuver_3', 'Gender_3', 'severity_3', 'Point of impact_3','id_vehicule_opposite_3']], on=['Num_Acc', 'id_vehicule'], how='left')

In [62]:
acci_clean['age_3'].fillna(0, inplace=True)
acci_clean['age_2'].fillna(0, inplace=True)

acci_clean.loc[acci_clean['number of involved vehicles']==1,'age_opposite_mean']= 0 
acci_clean.loc[acci_clean['number of involved vehicles']==2,'age_opposite_mean']=acci_clean['age_2']
acci_clean.loc[acci_clean['number of involved vehicles']>2,'age_opposite_mean']=(acci_clean['age_2']+acci_clean['age_3'])/2

/var/folders/y0/0nrj3m412p978185q3d2503sr02q24/T/ipykernel_5318/4166784484.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  acci_clean['age_3'].fillna(0, inplace=True)
/var/folders/y0/0nrj3m412p978185q3d2503sr02q24/T/ipykernel_5318/4166784484.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beha

In [63]:
acci_clean['vehicle_type_2'].fillna('No other vehcile', inplace=True)

/var/folders/y0/0nrj3m412p978185q3d2503sr02q24/T/ipykernel_5318/2450484712.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  acci_clean['vehicle_type_2'].fillna('No other vehcile', inplace=True)


In [64]:
passengers = acci_clean.loc[acci_clean['User category']=='Passenger']
drivers = acci_clean.loc[acci_clean['User category']=='Driver']

df_merged=pd.merge(passengers, drivers,on='id_vehicule', suffixes=('', '_driver'))

# Filter out rows where the id_usager is the same
df_filtered = df_merged[df_merged['id_usager'] != df_merged['id_usager_driver']]

# Select relevant columns
df_filtered = df_filtered[['id_usager', 'age_driver','Helmet_driver','Gender_driver','severity_driver']]

In [65]:
acci_clean['id_usager'].fillna(999,inplace=True)

/var/folders/y0/0nrj3m412p978185q3d2503sr02q24/T/ipykernel_5318/1015889634.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  acci_clean['id_usager'].fillna(999,inplace=True)


In [66]:

# Merge back with the original dataframe to include age_2, vehicle_type_2, and Maneuver_2
acci_clean = pd.merge(acci_clean, df_filtered[['id_usager', 'age_driver','Helmet_driver','Gender_driver','severity_driver']], how='left',on='id_usager')


In [67]:
acci_clean[['id_usager', 'age_driver','Helmet_driver','Gender_driver','severity_driver']]=acci_clean[['id_usager', 'age_driver','Helmet_driver','Gender_driver','severity_driver']].fillna('999')

In [68]:
acci_clean['Gender_driver'] = acci_clean['Gender_driver'].astype(str)
acci_clean['Helmet_driver'] = acci_clean['Helmet_driver'].astype(str)

In [69]:
bins = [0, 20, 40, 60,100]  # Intervalles d'âge
labels = ['0-20', '21-40', '41-60','61+']  # Étiquettes pour chaque intervalle

# Créer une nouvelle colonne 'Age category' contenant les catégories d'âge
acci_clean['Age category involved'] = pd.cut(acci_clean['age_2'], bins=bins, labels=labels, right=False).astype(str)

In [70]:
acci_clean['Accident location']='Other'
acci_clean.loc[acci_clean['situ'].isin([1]), 'Accident location'] = 'On road'
acci_clean.loc[acci_clean['situ'].isin([4]), 'Accident location'] = 'On sidewalk'
acci_clean.loc[acci_clean['situ'].isin([5]), 'Accident location'] = 'On cycle facility'

acci_clean.loc[acci_clean['situ'].isin([6]), 'Accident location'] = 'On other special way'


print( acci_clean['Accident location'].value_counts())

Accident location
On road                 17239
On cycle facility        6198
On other special way     1120
On sidewalk               714
Other                     389
Name: count, dtype: int64


In [71]:
# Group by 'Num_Acc' and count occurrences
count_per_acc = acci_clean.groupby('Num_Acc').size()

# Filter for counts equal to 1
unique_accs = count_per_acc[count_per_acc == 1].index

# Use .loc to filter the original DataFrame
filtered_df = acci_clean.loc[acci_clean['Num_Acc'].isin(unique_accs)]

## Load the final dataset

In [72]:
acci_clean.loc[acci_clean['Vehicle'].isin(['Pedestrian', 'E-scooter', 'E-bike', 'Bike'])].to_file('data.geojson')

In [73]:
acci_clean.to_csv('test_données.csv')